# Recommender Systems

Data: Matrix with ratings (with many missing values)

   columns: items
   rows: users
   
   $r_{ij}:$ rating of item $j$ by user $i$ (if it exists)
   
Goal: recommend items to users

Methods: item-based, user-based, matrix factorization-based

First, 

## Item-based

Two distinct methods: First, we'll count ratings, and second we'll use the rating values.

item -> the set of users that ahve rated the item

Similarity between sets:

   Jaccard Similarity: $J(A,B) = \frac{|A\cap B|}{|A\cup B|} $
      $J(A,B)\in [0,1]$ for all $A,B$
      
   Serendipity Similarity: $S(A,B) = \frac{|A\cap B|}{|B|}$
   $S(A,B)\in [0,1]$ for all $A,B$
   
 

In [1]:
import pandas as pd
import numpy as np

In [2]:
# load the movide ratings data
ratings = pd.read_csv('ratings_large.csv')

In [3]:
ratings

,userId,movieId,rating,timestamp,title
0,4,1,4.0,1113765937,Toy Story (1995)
1,4,2,4.0,1113767306,Jumanji (1995)
2,4,5,2.0,1123990453,Father of the Bride Part II (1995)
3,4,6,4.5,1113767242,Heat (1995)
4,4,10,4.0,1113765995,GoldenEye (1995)
...,...,...,...,...,...
10395149,283224,1388,2.0,851001995,Jaws 2 (1978)
10395150,283224,1394,3.0,851002354,Raising Arizona (1987)
10395151,283224,1396,3.0,851002354,Sneakers (1992)
10395152,283224,2019,5.0,851000812,Seven Samurai (Shichinin no samurai) (1954)


In [4]:
# users

ratings['userId'].nunique()

35289

In [5]:
# movies
ratings['title'].nunique()

1344

In [6]:
movie = "Monty Python's Life of Brian (1979)"

In [7]:
movie_sets = ratings.groupby('title')['userId'].apply(set)

In [8]:
# keep movies with at least 10 ratings
movie_sets = movie_sets[movie_sets.apply(len)>=10]

In [9]:
def Jaccard(setB):
    setA = movie_sets[movie]
    return len(setA.intersection(setB))/len(setA.union(setB))

In [10]:
# top 20 recommendations
movie_sets.apply(Jaccard).sort_values(ascending=False).head(20)

title
Monty Python's Life of Brian (1979)                                               1.000000
Monty Python and the Holy Grail (1975)                                            0.586490
Fish Called Wanda, A (1988)                                                       0.449349
Clockwork Orange, A (1971)                                                        0.445657
Blade Runner (1982)                                                               0.445490
2001: A Space Odyssey (1968)                                                      0.444474
Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)    0.439354
Star Wars: Episode IV - A New Hope (1977)                                         0.437053
Star Wars: Episode V - The Empire Strikes Back (1980)                             0.436310
Alien (1979)                                                                      0.435004
Back to the Future (1985)                                                         0.

In [11]:
#serendipity
def serendipity(setB):
    setA = movie_sets[movie]
    return len(setA.intersection(setB))/len(setB)

In [12]:
# top 20 recommendations
movie_sets.apply(serendipity).sort_values(ascending=False).head(20)

title
Monty Python's Life of Brian (1979)                                 1.000000
Monty Python's And Now for Something Completely Different (1971)    0.855663
Monty Python's The Meaning of Life (1983)                           0.845268
Return of the Pink Panther, The (1975)                              0.725561
History of the World: Part I (1981)                                 0.684424
Duck Soup (1933)                                                    0.679626
Sleeper (1973)                                                      0.675720
Producers, The (1968)                                               0.675028
Yellow Submarine (1968)                                             0.673108
Being There (1979)                                                  0.668960
Name of the Rose, The (Name der Rose, Der) (1986)                   0.661648
Seventh Seal, The (Sjunde inseglet, Det) (1957)                     0.658739
Pink Floyd: The Wall (1982)                                         0.

## Cosine Similarity

In [13]:
movie_ratings = ratings.pivot('userId','title')['rating']
movie_ratings.shape

(35289, 1344)

In [14]:
movie = 'Dark City (1998)' # sci-fi, similar to the matrix

movie1 = 'Matrix, The (1999)'

movie2 = 'Frozen (2013)'

In [15]:
x = movie_ratings.loc[:,movie]
y1 = movie_ratings.loc[:,movie1]
y2 = movie_ratings.loc[:,movie2]

In [16]:
# similarity between movie and movie1

np.sum(x*y1)/np.sqrt(np.sum(x**2)*np.sum(y1**2))

0.5357401524461298

In [17]:
# similarity between movie and movie2

np.sum(x*y2)/np.sqrt(np.sum(x**2)*np.sum(y2**2))

0.1377237178005576

In [18]:
def cosine_sim(colB):
    colA = movie_ratings.loc[:,movie]
    return np.sum(colA*colB)/np.sqrt(np.sum(colA**2)*np.sum(colB**2))

In [19]:
movie_ratings.apply(cosine_sim).sort_values(ascending=False).tail(20)

title
To Wong Foo, Thanks for Everything! Julie Newmar (1995)    0.128783
Wedding Planner, The (2001)                                0.128718
Now You See Me (2013)                                      0.127947
Circle of Friends (1995)                                   0.127890
Parent Trap, The (1961)                                    0.126986
Perks of Being a Wallflower, The (2012)                    0.126694
The Hunger Games: Mockingjay - Part 1 (2014)               0.125724
Miracle on 34th Street (1994)                              0.125107
13 Going on 30 (2004)                                      0.123044
Forget Paris (1995)                                        0.120409
Beverly Hillbillies, The (1993)                            0.117893
Little Princess, A (1995)                                  0.117308
Princess Diaries, The (2001)                               0.115596
Juror, The (1996)                                          0.114472
Help, The (2011)                          

## Correlation Similarity

In [20]:
def correlation_sim(colB):
    colA = movie_ratings.loc[:,movie]
    # mean ratings
    meanA = np.mean(colA)
    meanB = np.mean(colB)
    # norms
    normA = np.sum((colA - meanA)**2)
    normB = np.sum((colB-meanB)**2)
    
    # dot product
    dot = np.sum((colA-meanA)*(colB-meanB))
    
    return dot/np.sqrt(normA*normB)

In [21]:
movie_ratings.apply(correlation_sim).sort_values(ascending=False).head(20)

title
Dark City (1998)                                                   1.000000
Gattaca (1997)                                                     0.172906
Twelve Monkeys (a.k.a. 12 Monkeys) (1995)                          0.158235
Crow, The (1994)                                                   0.132797
Fifth Element, The (1997)                                          0.132455
Pi (1998)                                                          0.129861
City of Lost Children, The (Cité des enfants perdus, La) (1995)    0.124711
eXistenZ (1999)                                                    0.121718
Cube (1997)                                                        0.116946
Truman Show, The (1998)                                            0.115161
Blade Runner (1982)                                                0.115103
Sleepy Hollow (1999)                                               0.113087
Total Recall (1990)                                                0.110739
Army o

## User-Based Recommendation

notation: 
   $r_{ui} = $ rating of user $u$ for item $i$
   $I_{u} = $ set of items rated by user $u$
   
goal: predict $r_{uj}$ for $j\not\in I_u$ (items not rated by user $u$

user similarity function = correlation similarity. we'll use pandas 'corrwith'

For each item $j$, we'll use knn.

k-nearest neighbors: find the set of k users most similar to user u who have rated item j

idea: predicted $r_{uj}$ = average of the knn ratings

problem: different users may rate on different scale

solution: normalize:

$$
z_{uj} = \frac{r_{uj}-\mu_u}{\sigma_u}
$$

prediction: $r_{uj} = \mu_u +\sigma_u\frac{\sum_{}v\in knn_u(j)z_{vj}sim(u,v)}{\sum|sim(u,v)|}$

****

In [22]:
movie_ratings = ratings.pivot('title','userId')['rating']

user = 4 # example user selection

user_ratings = movie_ratings.loc[:,user]

In [23]:
user_ratings[user_ratings.isna()]

title
(500) Days of Summer (2009)                              NaN
10 Things I Hate About You (1999)                        NaN
101 Dalmatians (1996)                                    NaN
101 Dalmatians (One Hundred and One Dalmatians) (1961)   NaN
12 Years a Slave (2013)                                  NaN
                                                          ..
Young Guns (1988)                                        NaN
Zodiac (2007)                                            NaN
Zombieland (2009)                                        NaN
Zoolander (2001)                                         NaN
Zootopia (2016)                                          NaN
Name: 4, Length: 812, dtype: float64

In [24]:
# user mean rating
user_mean = user_ratings.mean()
user_mean

3.5253759398496243

In [25]:
# user std rating
user_std = user_ratings.std()
user_std

1.1591017244209465

In [26]:
# z-scores matrix
z_scores = (movie_ratings-movie_ratings.mean())/movie_ratings.std()

In [27]:
movie_ratings.drop(user,axis=1,inplace=True)

In [28]:
# user z-scores

user_zscores = (user_ratings-user_mean)/user_std

In [29]:
# similarities
sim = movie_ratings.corrwith(user_ratings)

/Users/vanmagnan/opt/anaconda3/lib/python3.8/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/vanmagnan/opt/anaconda3/lib/python3.8/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


In [30]:
# select number of neighbors
k=20
#select a movie
item = 'Duck Soup (1933)'

In [31]:
# k nearest neighbors similarities
knn_sim = sim[movie_ratings.loc[item].notna()==True].sort_values(ascending = False).head(k)

In [32]:
# k nearest neighbors
knn = knn_sim.index

In [33]:
# normalization factor
total_sim =knn_sim.abs().sum()

In [34]:
# knn z_scores
knn_z_scores = z_scores.loc[item,knn]

In [35]:
# prediction
prediction = user_mean + user_std*knn_z_scores.dot(knn_sim)/total_sim

In [36]:
print(item+': '+str(prediction.round(1)))

Duck Soup (1933): 4.2


In [37]:
unrated_movies = user_ratings[user_ratings.isna()].index.to_list()

In [38]:
for item in unrated_movies:
    # k nearest neighbors similarities
    knn_sim = sim[movie_ratings.loc[item].notna()==True].sort_values(ascending = False).head(k)
    
    # k nearest neighbors
    knn = knn_sim.index
    
    # normalization factor
    total_sim =knn_sim.abs().sum()
    
    # knn z_scores
    knn_z_scores = z_scores.loc[item,knn]
    # prediction
    prediction = user_mean + user_std*knn_z_scores.dot(knn_sim)/total_sim
    user_ratings.loc[item] = prediction

KeyError: 4

In [39]:
# top 20 recommendations
user_ratings.loc[unrated_movies].sort_values(ascending=False).head(20)

title
(500) Days of Summer (2009)                               3.941089
10 Things I Hate About You (1999)                              NaN
101 Dalmatians (1996)                                          NaN
101 Dalmatians (One Hundred and One Dalmatians) (1961)         NaN
12 Years a Slave (2013)                                        NaN
127 Hours (2010)                                               NaN
13th Warrior, The (1999)                                       NaN
20,000 Leagues Under the Sea (1954)                            NaN
2012 (2009)                                                    NaN
21 Grams (2003)                                                NaN
21 Jump Street (2012)                                          NaN
28 Days (2000)                                                 NaN
28 Days Later (2002)                                           NaN
3:10 to Yuma (2007)                                            NaN
40-Year-Old Virgin, The (2005)                          

##  Fake User Example

In [40]:
url = 'https://raw.githubusercontent.com/um-perez-alvaro/Data-Science-Theory/master/Data/fake_user.csv'

In [41]:
# fake user likes sci-fi movies, action-adventure, doesn't like romance or kid movies

user_ratings = pd.read_csv(url,index_col='title',squeeze=True)
user_ratings

title
(500) Days of Summer (2009)                              NaN
10 Things I Hate About You (1999)                        NaN
101 Dalmatians (1996)                                    NaN
101 Dalmatians (One Hundred and One Dalmatians) (1961)   NaN
12 Angry Men (1957)                                      NaN
                                                          ..
Zoolander (2001)                                         NaN
Zootopia (2016)                                          NaN
eXistenZ (1999)                                          NaN
xXx (2002)                                               NaN
¡Three Amigos! (1986)                                    NaN
Name: rating, Length: 1344, dtype: float64

In [42]:
sim = movie_ratings.corrwith(user_ratings)

In [43]:
unrated_movies = user_ratings[user_ratings.isna()].index.to_list()

In [44]:
for item in unrated_movies:
    # k nearest neighbors similarities
    knn_sim = sim[movie_ratings.loc[item].notna()==True].sort_values(ascending = False).head(k)
    
    # k nearest neighbors
    knn = knn_sim.index
    
    # normalization factor
    total_sim =knn_sim.abs().sum()
    
    # knn z_scores
    knn_z_scores = z_scores.loc[item,knn]
    # prediction
    prediction = user_mean + user_std*knn_z_scores.dot(knn_sim)/total_sim
    user_ratings.loc[item] = prediction

In [45]:
# top 20 recommendations
user_ratings.loc[unrated_movies].sort_values(ascending=False).head(20)

title
Lord of the Rings: The Return of the King, The (2003)                        4.657530
Great Escape, The (1963)                                                     4.643381
Usual Suspects, The (1995)                                                   4.628488
Once Upon a Time in the West (C'era una volta il West) (1968)                4.614946
Snatch (2000)                                                                4.583250
Léon: The Professional (a.k.a. The Professional) (Léon) (1994)               4.568253
Crow, The (1994)                                                             4.531541
Full Metal Jacket (1987)                                                     4.526120
Departed, The (2006)                                                         4.522565
Star Wars: Episode V - The Empire Strikes Back (1980)                        4.475611
Predator (1987)                                                              4.440852
Boondock Saints, The (2000)                     

In [46]:
# to create a user...
pd.DataFrame(index=movie_ratings.index, columns = ['rating']).to_csv('myuser.csv')

In [47]:
# create your ratings

In [48]:
# load your ratings
pd.read_csv('my_user.csv', index_col='title', squeeze=True)

FileNotFoundError: [Errno 2] No such file or directory: 'my_user.csv'

## The impact of the long tail

In [49]:
def weighted_corrwidth(y):
    
    x = user_ratings
    
    x_mean = np.mean(x)
    y_mean = np.mean(y)
    
    